# Hyperparam Sweep — Experiment Comparison

Auto-discover experiment folders from Google Drive, pick which runs to compare,
then visualise training curves and compute a scalar score (mean of averaged arena `avg` over ntuple matchups).

| Section | What happens |
|---|---|
| **§1** | Mount Drive, scan for `exp_*` folders, show interactive picker |
| **§2** | Load metrics + arena results for selected runs |
| **§3** | Training curves (loss, value stats, etc.) |
| **§4** | Arena comparison + scalar ranking |

<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/experiments/hyperparam-sweep/experiments/compare_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## §1. Discover & Select Experiments

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import json
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

MODELS_DIR = Path("/content/drive/MyDrive/models/" if IN_COLAB else "./")


def discover_experiments(models_dir: Path) -> list[dict]:
    """Scan for exp_* folders that contain 0_params.json."""
    experiments = []
    for d in sorted(models_dir.glob("exp_*")):
        if not d.is_dir():
            continue
        params_file = d / "0_params.json"
        if not params_file.exists():
            continue
        with params_file.open() as f:
            params = json.load(f)
        n_repeats_found = len(list(d.glob("repeat_*")))
        has_arena = (d / "0_arena_metrics.json").exists() or (
            n_repeats_found > 0
            and any(
                (d / f"repeat_{i}" / "0_arena_metrics.json").exists()
                for i in range(n_repeats_found)
            )
        )
        experiments.append({
            "path": str(d),
            "name": d.name,
            "params": params,
            "n_repeats_found": max(n_repeats_found, 1),
            "has_arena": has_arena,
        })
    return experiments


def make_label(exp: dict) -> str:
    """Human-readable one-liner for the picker."""
    p = exp["params"]
    parts = [exp["name"]]
    defaults = {
        "batch_size": 20000,
        "lr_initial": 3e-4,
        "lr_final": 1e-6,
        "lam": 0.7,
        "n_truncate": 5,
        "tau": 0.05,
        "epsilon": 0.1,
        "gradient_clip_max_norm": 0.1,
    }
    diffs = []
    for k, default_v in defaults.items():
        v = p.get(k)
        if v is not None and v != default_v:
            diffs.append(f"{k}={v}")
    if "epsilon_initial" in p:
        diffs.append(f"eps={p['epsilon_initial']}->{p['epsilon_final']}")
    if diffs:
        parts.append(" | ".join(diffs))
    arena_icon = "✅" if exp["has_arena"] else "⏳"
    parts.append(f"{arena_icon} {exp['n_repeats_found']}rep")
    return "  —  ".join(parts)


all_experiments = discover_experiments(MODELS_DIR)
print(f"Found {len(all_experiments)} experiment(s) in {MODELS_DIR}\n")

MIN_REPEATS_DEFAULT = 10

min_repeats_slider = widgets.IntSlider(
    value=MIN_REPEATS_DEFAULT,
    min=1,
    max=max((e["n_repeats_found"] for e in all_experiments), default=1),
    step=1,
    description="Min repeats:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

checkbox_container = widgets.VBox()
checkboxes: list[widgets.Checkbox] = []
experiments: list[dict] = []


def _rebuild_checkboxes(_=None):
    global checkboxes, experiments
    threshold = min_repeats_slider.value
    experiments = [e for e in all_experiments if e["n_repeats_found"] >= threshold]
    checkboxes = []
    for exp in experiments:
        cb = widgets.Checkbox(
            value=exp["has_arena"],
            description=make_label(exp),
            style={"description_width": "initial"},
            layout=widgets.Layout(width="100%"),
        )
        checkboxes.append(cb)
    checkbox_container.children = checkboxes
    print(
        f"\r{len(experiments)}/{len(all_experiments)} experiments "
        f"with >= {threshold} repeats",
        end="",
    )


select_all = widgets.Button(description="Select All", button_style="info")
select_none = widgets.Button(description="Deselect All", button_style="warning")


def _select_all(_):
    for cb in checkboxes:
        cb.value = True


def _select_none(_):
    for cb in checkboxes:
        cb.value = False


select_all.on_click(_select_all)
select_none.on_click(_select_none)
min_repeats_slider.observe(_rebuild_checkboxes, names="value")

display(min_repeats_slider)
display(widgets.HBox([select_all, select_none]))
_rebuild_checkboxes()
display(checkbox_container)

## §2. Load Selected Runs

Run this cell after checking the experiments you want to compare above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Build RUNS list from checkbox selection ─────────────────────────────────
RUNS: list[tuple[str, str]] = []
for cb, exp in zip(checkboxes, experiments):
    if cb.value:
        RUNS.append((exp["name"], exp["path"]))

print(f"Selected {len(RUNS)} run(s):")
for label, path in RUNS:
    print(f"  {label}: {path}")

In [ ]:
# ── Global plot settings ──────────────────────────────────────────────────────
X_AXIS: str = "step"
XLIM = None
YLIM = None

# ── Load training metrics (with repeat detection) ─────────────────────────────
runs_all: dict[str, list[list[dict]]] = {}
runs: dict[str, list[dict]] = {}

for label, folder in RUNS:
    folder_path = Path(folder)
    repeat_dirs = sorted(folder_path.glob("repeat_*"))
    if repeat_dirs:
        repeats = []
        for rd in repeat_dirs:
            p = rd / "0_metrics.json"
            if not p.exists():
                continue
            with p.open() as f:
                repeats.append(json.load(f))
        if not repeats:
            print(f"WARNING: No valid repeats in '{folder}', skipping '{label}'")
            continue
        runs_all[label] = repeats
        runs[label] = repeats[0]
        print(f"  {label}: {len(repeats)} repeats, {len(repeats[0])} entries each")
    else:
        p = folder_path / "0_metrics.json"
        if not p.exists():
            print(f"WARNING: {p} not found, skipping '{label}'")
            continue
        with p.open() as f:
            data = json.load(f)
        runs_all[label] = [data]
        runs[label] = data
        print(f"  {label}: {len(data)} entries")

print(f"\n{len(runs)} run(s) loaded.")

In [ ]:
# ── Load arena results ────────────────────────────────────────────────────────


def load_arena_results(folder):
    folder = Path(folder)
    compact = folder / "0_arena_metrics.json"
    if compact.exists():
        with compact.open() as f:
            data = json.load(f)
        return {entry["step"]: entry["aggregates"] for entry in data}
    return {}


def load_arena_results_all_repeats(folder):
    folder = Path(folder)
    repeat_dirs = sorted(folder.glob("repeat_*"))
    if repeat_dirs:
        return [load_arena_results(str(rd)) for rd in repeat_dirs]
    return [load_arena_results(str(folder))]


arena_runs_all: dict[str, list[dict]] = {}
arena_runs: dict[str, dict] = {}

for label, folder in RUNS:
    all_repeats = load_arena_results_all_repeats(folder)
    all_repeats = [r for r in all_repeats if r]
    if not all_repeats:
        print(f"  {label}: no arena results (skipping arena plots)")
        continue
    arena_runs_all[label] = all_repeats
    arena_runs[label] = all_repeats[0]
    steps = sorted(all_repeats[0].keys())
    n_rep = len(all_repeats)
    print(f"  {label}: {len(steps)} arena snapshots, {n_rep} repeat(s)")

print(f"\n{len(arena_runs)} run(s) with arena results.")

In [ ]:
# ── Parameters comparison table ───────────────────────────────────────────────
run_params = {}
for label, folder in RUNS:
    p = Path(folder) / "0_params.json"
    if p.exists():
        with p.open() as f:
            run_params[label] = json.load(f)

if run_params:
    df = pd.DataFrame(run_params).T
    df.index.name = "run"
    interesting = [
        "batch_size",
        "lr_initial",
        "lr_final",
        "gamma",
        "epsilon",
        "lam",
        "n_truncate",
        "tau",
        "use_gradient_clipping",
        "gradient_clip_max_norm",
        "n_steps",
        "n_repeats",
        "device",
    ]
    cols = [c for c in interesting if c in df.columns]
    display(df[cols].T)

## §3. Training Curves

In [ ]:
# ── Plotting helpers (from 2_plot_metrics.ipynb) ──────────────────────────────


def _normalized_auc(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    xf, yf = x[mask], y[mask]
    if len(xf) < 2:
        return None
    x_range = xf[-1] - xf[0]
    if x_range == 0:
        return None
    return float(np.trapezoid(yf, xf) / x_range)


def _format_nauc(val):
    return "n/a" if val is None else f"{val:.4g}"


def _resolve_x_axis(x_axis):
    return x_axis if x_axis is not None else X_AXIS


def get_x_axis(data, x_axis=None):
    mode = _resolve_x_axis(x_axis)
    if mode == "time":
        x = (
            np.array([m.get("training_elapsed_s", 0.0) for m in data], dtype=np.float64)
            / 3600.0
        )
        return x, "elapsed time (h)"
    return np.array([m["step"] for m in data]), "step"


def get_metric_with_stats(label, key, x_axis=None):
    repeats = runs_all[label]
    n_rep = len(repeats)
    x, x_label = get_x_axis(repeats[0], x_axis)
    min_len = min(len(r) for r in repeats)
    x = x[:min_len]
    ys = np.array([[m[key] for m in r[:min_len]] for r in repeats], dtype=np.float64)
    y_mean = ys.mean(axis=0)
    y_std = ys.std(axis=0, ddof=1) if n_rep > 1 else np.zeros_like(y_mean)
    return x, y_mean, y_std, x_label, n_rep


def _apply_limits(ax, xlim=None, ylim=None):
    xl = xlim if xlim is not None else XLIM
    yl = ylim if ylim is not None else YLIM
    if xl is not None:
        ax.set_xlim(*xl)
    if yl is not None:
        ax.set_ylim(*yl)


def plot_metric(
    metric_key,
    *,
    x_axis=None,
    title=None,
    ylabel=None,
    yscale="linear",
    xlim=None,
    ylim=None,
    figsize=(12, 5),
):
    fig, ax = plt.subplots(figsize=figsize)
    x_label_auto = "step"
    for label in runs_all:
        x, y_mean, y_std, x_label_auto, n_rep = get_metric_with_stats(
            label, metric_key, x_axis
        )
        mask = np.isfinite(y_mean)
        x, y_mean, y_std = x[mask], y_mean[mask], y_std[mask]
        nauc = _normalized_auc(x, y_mean)
        rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""
        (line,) = ax.plot(
            x,
            y_mean,
            marker=".",
            markersize=3,
            linewidth=1,
            label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
        )
        if n_rep > 1:
            ax.fill_between(
                x, y_mean - y_std, y_mean + y_std, alpha=0.2, color=line.get_color()
            )
    ax.set_xlabel(x_label_auto)
    ax.set_ylabel(ylabel or metric_key)
    ax.set_title(title or metric_key)
    ax.set_yscale(yscale)
    _apply_limits(ax, xlim, ylim)
    ax.legend(fontsize="small")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_metric("loss", title="Training Loss (MSE)", ylabel="loss")

In [ ]:
plot_metric(
    "rel_weight_update",
    title="Relative Weight Update ||dW|| / ||W||",
    ylabel="||dW|| / ||W||",
    yscale="log",
)

## §4. Arena Comparison & Scalar Ranking

In [ ]:
# ── Arena helpers ─────────────────────────────────────────────────────────────


def get_matchup_key(row):
    return f"{row['agent_yellow']} vs {row['agent_red']}"


def list_matchups(arena_data):
    matchups = set()
    for rows in arena_data.values():
        for r in rows:
            if r["epsilon_yellow"] == 0.0 and r["epsilon_red"] == 0.0:
                matchups.add(get_matchup_key(r))
    return sorted(matchups)


def extract_series(arena_data, matchup):
    series = {"steps": []}
    for step, rows in sorted(arena_data.items()):
        for r in rows:
            if r["epsilon_yellow"] != 0.0 or r["epsilon_red"] != 0.0:
                continue
            if get_matchup_key(r) != matchup:
                continue
            series["steps"].append(step)
            for k in (
                "games",
                "yellow_wins",
                "red_wins",
                "draws",
                "score",
                "avg",
                "timeouts",
                "illegal_moves",
                "exceptions",
                "total_time_s",
            ):
                series.setdefault(k, []).append(r.get(k, 0))
            break
    return series


def _step_to_time_map(label):
    if label not in runs:
        return {}
    return {
        int(m["step"]): m.get("training_elapsed_s", 0.0) / 3600.0 for m in runs[label]
    }


def _map_steps_to_x(steps, label, x_axis=None):
    mode = _resolve_x_axis(x_axis)
    if mode == "time":
        mapping = _step_to_time_map(label)
        if mapping:
            known_steps = sorted(mapping.keys())
            known_times = [mapping[s] for s in known_steps]
            x = list(np.interp(steps, known_steps, known_times))
        else:
            x = [float(s) for s in steps]
        return x, "elapsed time (h)"
    return steps, "step"

In [ ]:
def plot_arena_averaged(
    *,
    starts_with="ntuple",
    ends_with=None,
    metric="avg",
    x_axis=None,
    title=None,
    ylabel=None,
    xlim=None,
    ylim=None,
    figsize=(14, 6),
    use_bootstrap=False,
):
    """Average a metric over all matching matchups and plot one line per run.

    Returns: dict[label, y_mean_array] for scalar scoring.
    """
    ref_label = list(arena_runs_all.keys())[-1]
    all_matchups = list_matchups(arena_runs_all[ref_label][0])

    filtered = [
        m
        for m in all_matchups
        if (not starts_with or m.startswith(starts_with))
        and (not ends_with or m.endswith(ends_with))
    ]

    if not filtered:
        print("No matching matchups found.")
        return None

    fig, ax = plt.subplots(figsize=figsize)
    x_label = "step"
    averaged_results = {}

    for label in arena_runs_all:
        all_repeats = arena_runs_all[label]
        n_rep = len(all_repeats)

        per_repeat_averaged = []
        ref_steps = None

        for arena_data in all_repeats:
            matchup_series = []
            for m in filtered:
                s = extract_series(arena_data, m)
                if s["steps"] and metric in s:
                    matchup_series.append(s[metric])
                    if ref_steps is None:
                        ref_steps = s["steps"]
            if not matchup_series:
                continue
            min_len = min(len(ms) for ms in matchup_series)
            arrs = [np.array(ms[:min_len]) for ms in matchup_series]
            per_repeat_averaged.append(np.mean(arrs, axis=0))

        if not per_repeat_averaged or ref_steps is None:
            continue

        min_len = min(len(a) for a in per_repeat_averaged)
        steps = ref_steps[:min_len]
        ys = np.array([a[:min_len] for a in per_repeat_averaged], dtype=np.float64)
        y_mean = ys.mean(axis=0)
        y_std = ys.std(axis=0, ddof=1) if n_rep > 1 else np.zeros_like(y_mean)

        x, x_label = _map_steps_to_x(steps, label, x_axis)
        xa = np.asarray(x, dtype=np.float64)
        nauc = _normalized_auc(xa, y_mean)
        rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""

        (line,) = ax.plot(
            xa,
            y_mean,
            marker="o",
            markersize=4,
            linewidth=1.5,
            label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
        )
        if n_rep > 1:
            ax.fill_between(
                xa, y_mean - y_std, y_mean + y_std, alpha=0.2, color=line.get_color()
            )

        averaged_results[label] = y_mean

    filt = []
    if starts_with:
        filt.append(f"starts_with={starts_with!r}")
    if ends_with:
        filt.append(f"ends_with={ends_with!r}")
    filter_desc = f" ({', '.join(filt)})" if filt else ""

    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel or f"averaged {metric}")
    ax.set_title(title or f"Averaged Arena {metric}{filter_desc}")
    _apply_limits(ax, xlim, ylim)
    ax.legend(fontsize="small")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    return averaged_results

In [ ]:
# ── Averaged arena score plot + scalar ranking ─────────────────────────────────
avg_results = plot_arena_averaged(starts_with="ntuple")

In [ ]:
# ── Scalar ranking table ──────────────────────────────────────────────────────
if avg_results:
    scores = {k: v.mean() for k, v in avg_results.items()}
    ranking = pd.DataFrame(
        sorted(scores.items(), key=lambda x: x[1], reverse=True),
        columns=["Experiment", "Mean Avg Score (ntuple matchups)"],
    )
    ranking.index = range(1, len(ranking) + 1)
    ranking.index.name = "Rank"
    display(ranking)
else:
    print("No arena results to rank.")

In [ ]:
# ── Bar chart of scalar scores ────────────────────────────────────────────────
if avg_results:
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    labels_sorted = [s[0] for s in sorted_scores]
    values_sorted = [s[1] for s in sorted_scores]

    fig, ax = plt.subplots(figsize=(12, max(4, len(labels_sorted) * 0.5)))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(labels_sorted)))
    bars = ax.barh(labels_sorted[::-1], values_sorted[::-1], color=colors[::-1])
    ax.set_xlabel("Mean Avg Score (ntuple matchups)")
    ax.set_title("Experiment Ranking")
    ax.grid(True, alpha=0.3, axis="x")
    for bar, val in zip(bars, values_sorted[::-1]):
        ax.text(
            bar.get_width() + 0.005,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}",
            va="center",
            fontsize=9,
        )
    plt.tight_layout()
    plt.show()

## §5. Per-Matchup Drill-Down

Compare runs for individual matchups.

In [ ]:
def extract_series_with_stats(label, matchup, metric):
    all_repeats = arena_runs_all[label]
    n_rep = len(all_repeats)
    all_series = []
    ref_steps = None
    for arena_data in all_repeats:
        s = extract_series(arena_data, matchup)
        if not s["steps"]:
            continue
        all_series.append(s)
        if ref_steps is None:
            ref_steps = s["steps"]
    if not all_series:
        return [], np.array([]), np.array([]), 0
    min_len = min(len(s[metric]) for s in all_series)
    steps = ref_steps[:min_len]
    ys = np.array([s[metric][:min_len] for s in all_series], dtype=np.float64)
    y_mean = ys.mean(axis=0)
    y_std = ys.std(axis=0, ddof=1) if n_rep > 1 else np.zeros_like(y_mean)
    return steps, y_mean, y_std, len(all_series)


def plot_arena_compare_runs(
    matchup,
    metric="avg",
    *,
    x_axis=None,
    title=None,
    ylabel=None,
    xlim=None,
    ylim=None,
    figsize=(12, 5),
):
    fig, ax = plt.subplots(figsize=figsize)
    x_label = "step"
    for label in arena_runs_all:
        steps, y_mean, y_std, n_rep = extract_series_with_stats(label, matchup, metric)
        if not steps:
            continue
        x, x_label = _map_steps_to_x(steps, label, x_axis)
        xa, ya = np.asarray(x, dtype=np.float64), y_mean
        nauc = _normalized_auc(xa, ya)
        rep_suffix = f" (n={n_rep})" if n_rep > 1 else ""
        if n_rep > 1:
            ax.errorbar(
                xa,
                ya,
                yerr=y_std,
                marker="o",
                markersize=4,
                linewidth=1.5,
                capsize=3,
                label=f"{label}{rep_suffix} [nAUC={_format_nauc(nauc)}]",
            )
        else:
            ax.plot(
                xa,
                ya,
                marker="o",
                markersize=4,
                linewidth=1.5,
                label=f"{label} [nAUC={_format_nauc(nauc)}]",
            )
    ax.set_xlabel(x_label)
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or f"{matchup} — {metric}")
    _apply_limits(ax, xlim, ylim)
    ax.axhline(y=0.0, color="gray", linestyle="--", alpha=0.4)
    ax.legend(fontsize="small")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Available matchups
if arena_runs:
    ref = list(arena_runs.keys())[-1]
    print(f"Matchups in '{ref}':")
    for m in list_matchups(arena_runs[ref]):
        print(f"  - {m}")

In [ ]:
plot_arena_compare_runs("ntuple vs bitbully-full-strength", "avg")
plot_arena_compare_runs("ntuple vs bitbully-16ply-book12ply", "avg")
plot_arena_compare_runs("ntuple vs random", "avg")

## §6. Export Summary for Next Experiment Design

Run this cell to write a compact JSON summary of all selected experiments (params + scalar scores + per-matchup final values). Hand the output file to Claude to design the next batch.

In [ ]:
# ── Export summary JSON ───────────────────────────────────────────────────────
EXPORT_PATH = MODELS_DIR / "sweep_summary.json"

summary = {
    "baseline_defaults": {
        "n_steps": 25_000,
        "n_evaluate": 1_000,
        "n_repeats": 10,
        "batch_size": 20_000,
        "epsilon": 0.1,
        "lr_initial": 3e-4,
        "lr_final": 1e-6,
        "gamma": 0.99999,
        "use_gradient_clipping": True,
        "gradient_clip_max_norm": 0.1,
        "optimizer_betas": [0.9, 0.999],
        "optimizer_eps": 1e-8,
        "optimizer_weight_decay": 0,
        "lam": 0.7,
        "n_truncate": 5,
        "use_target_net": True,
        "use_online_net_for_action": True,
        "tau": 0.05,
    },
    "experiments": {},
}

for label, folder in RUNS:
    entry = {"folder": folder}

    # Params
    if label in run_params:
        entry["params"] = run_params[label]

    # Repeats info
    if label in runs_all:
        entry["n_repeats"] = len(runs_all[label])

    # Scalar score
    if avg_results and label in avg_results:
        entry["scalar_score"] = float(avg_results[label].mean())

    # nAUC of averaged arena curve
    if avg_results and label in avg_results and label in arena_runs_all:
        ref_arena = arena_runs_all[label][0]
        ref_matchups = list_matchups(ref_arena)
        ntuple_matchups = [m for m in ref_matchups if m.startswith("ntuple")]
        if ntuple_matchups:
            s0 = extract_series(ref_arena, ntuple_matchups[0])
            if s0["steps"]:
                xa = np.asarray(s0["steps"], dtype=np.float64)
                ya = avg_results[label]
                min_len = min(len(xa), len(ya))
                nauc = _normalized_auc(xa[:min_len], ya[:min_len])
                if nauc is not None:
                    entry["nAUC"] = round(nauc, 6)

    # Per-matchup final avg (last evaluation step, averaged over repeats)
    if label in arena_runs_all:
        matchup_finals = {}
        ref_arena = arena_runs_all[label][0]
        for matchup in list_matchups(ref_arena):
            steps, y_mean, y_std, n_rep = extract_series_with_stats(
                label, matchup, "avg"
            )
            if len(y_mean) > 0:
                matchup_finals[matchup] = {
                    "final_avg": round(float(y_mean[-1]), 4),
                    "final_std": round(float(y_std[-1]), 4),
                }
        entry["matchup_finals"] = matchup_finals

    # Diffs from baseline
    if label in run_params:
        diffs = {}
        for k, default_v in summary["baseline_defaults"].items():
            v = run_params[label].get(k)
            if v is not None and v != default_v:
                diffs[k] = v
        for extra in ("epsilon_initial", "epsilon_final", "epsilon_schedule"):
            if extra in run_params[label]:
                diffs[extra] = run_params[label][extra]
        entry["diffs_from_baseline"] = diffs

    summary["experiments"][label] = entry

# Sort by score
if any("scalar_score" in e for e in summary["experiments"].values()):
    summary["ranking"] = sorted(
        [
            (label, e["scalar_score"])
            for label, e in summary["experiments"].items()
            if "scalar_score" in e
        ],
        key=lambda x: x[1],
        reverse=True,
    )

with EXPORT_PATH.open("w") as f:
    json.dump(summary, f, indent=2)

print(f"Summary written to {EXPORT_PATH}")
print(
    f"({len(summary['experiments'])} experiments, {EXPORT_PATH.stat().st_size} bytes)"
)
print("\nCopy this file and paste its contents when starting a new Claude session.")